# MERFISH 300k 2 Gene Panels

- Creator: Sebastian Birk (sb75@sanger.ac.uk)
- Date of Creation: 25.09.2024
- Date of Last Modification: 25.09.2024 (by Sebastian Birk; sb75@sanger.ac.uk)

# 1. Setup

## 1.1 Load Libraries 

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
sys.path.append("../../utils")

In [ ]:
import logging
import os
import pickle
import random
from datetime import datetime

import anndata as ad
import torch
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scib_metrics
import seaborn as sns
from sklearn.model_selection import train_test_split

from nichejepa.datasets import prepare_dataset
from nichejepa.infer import infer
from nichejepa.train import train
from nichejepa.utils.config import create_params_from_YAML_wandb_config
from nichejepa.utils.distributed import init_distributed
from nichejepa.utils.evaluation import clustering_metrics, classification_metrics

from analysis_utils import *

## 1.2 Load Config/Params
All setup will be done in the config file. Please edit that file to continue.

In [ ]:
dataset = 'merfish_300k_2_gene_panels'

# Set inference params
load_timestamp = None # '25092024_162214_058', '25092024_152307_874', None
read_checkpoint = None # 'jepa-ep5.pth.tar', None
exclude_unshared_tokens = False # True, False
agg_type = 'cls' # 'avg', 'weighted_avg', 'cls'

In [ ]:
# Set up logger
rank=0
logging.basicConfig()
logger = logging.getLogger()
logger.setLevel(logging.INFO if rank == 0 else logging.ERROR)

# Load config
if not load_timestamp:
    params = create_params_from_YAML_wandb_config(f'../../../nichejepa/configs/{dataset}.yaml',logger)
else:
    params = create_params_from_YAML_wandb_config(f'../../artifacts/{dataset}/{load_timestamp}/params.yaml',logger)

if read_checkpoint:
    params['meta']['read_checkpoint'] = read_checkpoint

params

In [ ]:
if not load_timestamp:
    artifact_folder_path = '../../artifacts'
    current_timestamp = (
        datetime.now().strftime("%d%m%Y_%H%M%S") +
        f"_{datetime.now().microsecond // 1000:03d}")
    folder_path = os.path.join(artifact_folder_path,
                               params['data']['data_set_name'],
                               current_timestamp)
else:
    folder_path = f'../../artifacts/{dataset}/{load_timestamp}'
    
    folder_path

## 1.3 Create Train & Test Dataset

In [ ]:
train_dataset, test_dataset = prepare_dataset(params)
train_dataset

## 1.4 Run INIT Distributed

In [ ]:
devices=['cuda:0']
world_size=1
os.environ['CUDA_VISIBLE_DEVICES'] = str(devices[rank].split(':')[-1])

logging.basicConfig()
logger = logging.getLogger()
logger.setLevel(logging.INFO if rank == 0 else logging.ERROR)

world_size, rank = init_distributed(rank_and_world_size=(rank, world_size), port=random.randint(40000, 50000))
logger.info(f'Running... (rank: {rank}/{world_size})')

# 2. Model Training

In [ ]:
### Train model
train(params,
      train_dataset,
      test_dataset,
      save_folder_path=folder_path)

INFO:root:[3,  6330] loss: 0.224 masks: 93.8 18.6 [wd: 5.73e-02] [lr: 2.00e-05] [mem: 3.79e+03] (52.8 ms)
INFO:root:[3,  6330] grad_stats: [1.57e-02 7.97e-03] (6.59e-04, 1.57e-02)
INFO:root:[3,  6340] loss: 0.224 masks: 93.8 18.6 [wd: 5.73e-02] [lr: 2.00e-05] [mem: 3.79e+03] (52.9 ms)
INFO:root:[3,  6340] grad_stats: [2.17e-02 1.18e-02] (9.90e-04, 2.17e-02)
INFO:root:[3,  6350] loss: 0.224 masks: 93.8 18.6 [wd: 5.73e-02] [lr: 2.00e-05] [mem: 3.79e+03] (52.9 ms)
INFO:root:[3,  6350] grad_stats: [1.53e-02 9.11e-03] (7.81e-04, 1.53e-02)
INFO:root:[3,  6360] loss: 0.224 masks: 93.8 18.6 [wd: 5.73e-02] [lr: 2.00e-05] [mem: 3.79e+03] (52.9 ms)
INFO:root:[3,  6360] grad_stats: [1.66e-02 9.10e-03] (7.54e-04, 1.66e-02)
INFO:root:[3,  6370] loss: 0.224 masks: 93.8 18.6 [wd: 5.73e-02] [lr: 2.00e-05] [mem: 3.79e+03] (52.9 ms)
INFO:root:[3,  6370] grad_stats: [1.47e-02 9.33e-03] (8.77e-04, 1.47e-02)
INFO:root:[3,  6380] loss: 0.224 masks: 93.8 18.6 [wd: 5.73e-02] [lr: 2.00e-05] [mem: 3.79e+03] (52.

# 3. Analysis

## 3.1 Run Model Inference

In [ ]:
# Get gene tokens that overlap between gene panels to obtain excluded tokens from aggregation
if exclude_unshared_tokens:
    gene_panel_gene_mapping_file_path = '/lustre/scratch126/cellgen/team361/am84/NicheJEPA/datasets/am84/tokenizer/cell-neighborhood-tokenizer/merfish_mouse_brain_300k_2-gene-panels/gene_panel_ID_to_gene_panel_dict.pkl'
    token_dictionary_file_path = '/lustre/scratch126/cellgen/team361/am84/NicheJEPA/datasets/am84/tokenizer/cell-neighborhood-tokenizer/merfish_mouse_brain_300k_2-gene-panels/token_dictionary_no_cls_no_sep.pkl'
    
    with open(gene_panel_gene_mapping_file_path, 'rb') as file:
        gene_panel_gene_mapping = pickle.load(file)
    
    with open(token_dictionary_file_path, 'rb') as file:
        token_dictionary = pickle.load(file)
    
    genes_panel0 = set(gene_panel_gene_mapping[0])
    genes_panel1 = set(gene_panel_gene_mapping[1])
    
    overlapping_genes = list(genes_panel0.intersection(genes_panel1))
    
    overlapping_gene_tokens = [token_dictionary[gene] for gene in overlapping_genes if gene in token_dictionary]
    
    excluded_tokens = list(set(token_dictionary.values()) - set(overlapping_gene_tokens))

In [ ]:
# Run model inference
adata = infer(params,
              train_dataset,
              load_folder_path=folder_path,
              agg_type=agg_type,
              agg_excluded_tokens=excluded_tokens if exclude_unshared_tokens else None,
              cell_gene_ids=[],
              neighborhood_gene_ids=[])

# Save adata
adata.write(f'{folder_path}/adata_{agg_type}'
            f'{"_shared_genes" if exclude_unshared_tokens and agg_type != "cls" else ("_all_genes" if agg_type != "cls" else "")}.h5ad')

## 3.2 Plot Embeddings

In [ ]:
adata.obs

In [ ]:
layer = params['meta']['enc_depth'] - 1

In [ ]:
# Compute UMAP representation for (aggregated) neighborhood gene embeddings
sc.pp.neighbors(adata,
                n_neighbors=15,
                use_rep=f'neighborhood_emb_layer_{layer}',
                key_added='neighborhood')
sc.tl.umap(adata,
           neighbors_key='neighborhood')

os.makedirs(f'{folder_path}/figures', exist_ok=True)

labels = ['dataset_id', 'major_brain_region', 'batch_id']

# Plot (aggregated) neighborhood gene embeddings, colored by labels
for label in labels:
    sc.pl.umap(adata,
               color=label,
               show=False)
    plt.savefig(f'{folder_path}/figures/{label}_umap_{agg_type}'
                f'{"_shared_genes" if exclude_unshared_tokens and agg_type != "cls" else ("_all_genes" if agg_type != "cls" else "")}.svg',
                dpi=300,
                bbox_inches='tight')

## 3.3 Plot Embedding Similarity Stats

## 3.4 Compute Metrics

In [ ]:
os.makedirs(f'{folder_path}/metrics', exist_ok=True)

labels = ['cell_types', 'major_brain_region']

# NMI & ARI for label cols (neighborhood embedding)
for label in labels:
    cell_type_metrics = clustering_metrics(
        adata,
        emb_key=f'neighborhood_emb_layer_{layer}',
        label_col=label)
    
    metrics_df = pd.DataFrame([cell_type_metrics])
    metrics_df.to_csv(f'{folder_path}/metrics/{label}_metrics{"_shared_genes" if exclude_unshared_tokens and agg_type != "cls" else ("_all_genes" if agg_type != "cls" else "")}.csv', index=False)

## 3.5 Plot Loss Curve

In [ ]:
df = pd.read_csv(f'{folder_path}/jepa_r0.csv')
df['loss'] = pd.to_numeric(df['loss'], errors='coerce')

plt.figure(figsize=(10, 6))
plt.plot(df.index,df['loss'])
plt.title('Loss Function Over Iterations')
plt.xlabel('Iteration')
plt.ylabel('Loss')
plt.grid(True)
plt.show()